In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os, glob
from esda.moran import Moran
import libpysal
import warnings
warnings.filterwarnings("ignore")

In [ ]:
ours_path = "../../data/regression_outputs/regmodels_spatial_self/Sampling_kcenter/*/*/Fuse/Token_Concat_spatial_self_Spatial/results.csv"
no_geo_path = "../../data/regression_outputs/regmodels_spatial_self/Ratio_no_geo/*/*/Fuse/results.csv"

In [ ]:
def get_df(path, column_name):
    files = glob.glob(path)
    print(len(files))
    df_list = []

    for file in files:
        tmp_df = pd.read_csv(file)
        if 'no_geo' in file:
            country = file.split('/')[-4]
            city = file.split('/')[-3]
        else:
            country = file.split('/')[-5]
            city = file.split('/')[-4]
        label_sdg_file = f"../../data/processed/0labels/{country}.csv"
        labels_sdg = pd.read_csv(label_sdg_file)
        tmp_df = pd.merge(tmp_df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
        tmp_df['country'] = country
        tmp_df['city'] = city
        df_list.append(tmp_df)
    df = pd.concat(df_list, ignore_index=True)
    if 'k' in df.columns:
        df.drop(['k', 'ID'], axis=1, inplace=True)
    else:
        df.drop(['ID',], axis=1, inplace=True)
    # df = df.groupby(['ratio', 'SDG']).mean().reset_index()
    df = df[['ratio','target', 'SDG', 'all_r2', 'country', 'city']]
    df.columns = ['ratio', 'target', 'SDG', column_name, 'country', 'city']
    return df

In [ ]:
ours_df = get_df(ours_path, 'Ours')
no_geo_df = get_df(no_geo_path, 'Non-spatial')
df = pd.merge(ours_df, no_geo_df, on=['ratio', 'target', 'SDG', 'country', 'city'])
df['ratio'] = df['ratio']*100
df['delta'] = df['Ours'] - df['Non-spatial']
df

In [ ]:
US_TARGETS="logcrime,logpetty,walkbike_per_cbg,publictrans_per_cbg,drove_alone_per_cbg,estvmiles,estpmiles,estvtrp,estptrp,obesitycru,diabetescr,lpacrudepr,mhlthcrude,phlthcrude,cancercrud,logincome,povertyline_below100,povertyline_below200"
US_TARGETS=US_TARGETS.split(',')
AU_TARGETS="med_hhinc,age65,arthritis,asthma,cancer,diabetes,heart_disease,kidney_disease,lung_condition,mental_health,edu_year12,edu_noschool,med_capgain,mean_capgain,unemploy,internet,popden,renter,publictrans,drive,bike,walk"
AU_TARGETS=AU_TARGETS.split(',')
BR_TARGETS="BR01,BR02,BR12,BR13,BR14,BR17"
BR_TARGETS=BR_TARGETS.split(',')
CH_TARGETS="HK02,HK06,HK07,HK15,HK17"
CH_TARGETS=CH_TARGETS.split(',')
FR_TARGETS="FR02,FR03,FR04,FR08,FR09,FR11"
FR_TARGETS=FR_TARGETS.split(',')
PT_TARGETS="PT16,PT24,PT28,PT33,PT34,PT37,PT38,PT35"
PT_TARGETS=PT_TARGETS.split(',')
NG_TARGETS="market_den,pub_health,health_den"
NG_TARGETS=NG_TARGETS.split(',')

countries_targets = {
    'US': US_TARGETS,
    'Australia': AU_TARGETS,
    'Brazil': BR_TARGETS,
    'China': CH_TARGETS,
    'France': FR_TARGETS,
    'Portugal': PT_TARGETS,
    'Nigeria': NG_TARGETS
}

In [ ]:
def cal_moransi(geo_file):
    country = geo_file.split('/')[-3]
    city = geo_file.split('/')[-2]
    targets = countries_targets[country]
    
    geo_df = pd.read_pickle(geo_file)
    geo_df = gpd.GeoDataFrame(geo_df, geometry='geometry')
    
    # Ensure geometry is valid.
    geo_df = geo_df[geo_df.geometry.notna()]
    
    valid_targets = set(geo_df.columns) & set(targets)
    # Keep only geometry + valid target columns to avoid SettingWithCopyWarning.
    geo_df = geo_df[['GEOID'] + list(valid_targets) + ['geometry']].copy()

    moransi_list = []
    
    for variable in valid_targets:
        # Drop rows where the current variable is NaN.
        gdf = geo_df.dropna(subset=[variable]).copy()
        
        # ---------------------------------------------------------
        # Key check: enough rows to compute neighbors + correlation?
        # ---------------------------------------------------------
        # Skip if fewer than 2 rows (no neighbors / correlation possible).
        if len(gdf) < 5: 
            # print(f"Skipping {variable} in {city}: Not enough data (n={len(gdf)})")
            continue
        
        try:
            # Suppress the islands warning while building weights.
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                # Build the spatial-weights matrix.
                w = libpysal.weights.KNN.from_dataframe(gdf, k=6)
                
                # Check whether every polygon is an island.
                # Moran's I is undefined if there are no neighbors.
                if w.n == w.islands:
                    # print(f"Skipping {variable}: All polygons are islands")
                    continue
                
                w.transform = 'r'
                y = gdf[variable].values
                
                moran = Moran(y, w)
                
                moransi_list.append([variable, moran.I, moran.EI, moran.p_sim])
                
        except Exception as e:
            # Catch other topology errors so they don't break the loop.
            print(f"Error processing {city} - {variable}: {str(e)}")
            continue

    moransi_df = pd.DataFrame(moransi_list, columns=['target', 'Moran_I', 'Expected_I', 'p_value'])
    moransi_df['country'] = country
    moransi_df['city'] = city
    
    return moransi_df

In [ ]:
geo_files = glob.glob("../../data/processed/*/*/labels.pkl")
moransi_list = []
for geo_file in geo_files:
    moransi_df = cal_moransi(geo_file)
    moransi_list.append(moransi_df)
moransi_df = pd.concat(moransi_list, ignore_index=True)
moransi_df

In [ ]:
df = pd.merge(df, moransi_df, on=['target', 'country', 'city'])
df

In [ ]:
df.to_csv("../../data/scripts_sdg/0figure_data_prepare/fig2_geo_reg_moransi_results.csv", index=False)